# ⚖️ Métodos de balanceo: Logit Adjustment y LDAM (Mac / MPS)

Prueba dos métodos documentados de balanceo de clases sobre tu mejor modelo (RoBERTa clínico, auditor de 7 clases), manteniendo todo lo demás idéntico al baseline para una comparación limpia.

**Métodos incluidos:**
- **Logit Adjustment** (Menon et al., 2021): durante el entrenamiento suma `tau · log(prior)` a los logits, forzando al modelo a calibrar hacia una distribución balanceada. En inferencia se usan los logits crudos. Reemplaza a la ponderación de clases (no se usan ambos a la vez).
- **LDAM** (Cao et al., 2019): asigna **márgenes mayores** a las clases raras (margen ∝ 1/n^¼) y opcionalmente activa reponderación diferida (DRW) en épocas tardías.

**Baseline a comparar:** RoBERTa clínico con cross-entropy ponderada → F1-macro **0,5871** · AUC zona **0,93**.

⚠️ Expectativa honesta: con solo 73 casos de riesgo, estos métodos reordenan la misma señal; el cambio esperable es pequeño (±0,02). El valor principal es completar la comparación metodológica.

---
**Uso:** colocar en `notebooks/`. Lee `../data/processed/dataset_clean.csv`. Restart + Run All. Corre los dos métodos (celdas 7 y 8) y compara en la celda 9.

## 1 · Configuración

In [1]:
import numpy as np, pandas as pd, random
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, classification_report, roc_auc_score, confusion_matrix

SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE='cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
MODELO='PlanTL-GOB-ES/roberta-base-biomedical-clinical-es'; MODELO_ALT='BSC-LT/roberta-base-biomedical-clinical-es'
MAX_LENGTH=256; LR=3e-5; EPOCHS=15; PATIENCE=4; BATCH=16
print("Dispositivo:", DEVICE, "| max_length:", MAX_LENGTH)

Dispositivo: mps | max_length: 256


## 2 · Datos y split (idéntico al baseline)

In [2]:
df=pd.read_csv('../data/processed/dataset_clean.csv', encoding='utf-8')
X=df['auditor_input'].values; y=df['BI-RADS'].values
Xtv,Xte,ytv,yte=train_test_split(X,y,test_size=0.15,random_state=SEED,stratify=y)
Xtr,Xva,ytr,yva=train_test_split(Xtv,ytv,test_size=0.176,random_state=SEED,stratify=ytv)
print(f"Train {len(Xtr)} | Val {len(Xva)} | Test {len(Xte)} (riesgo: {(yte>=4).sum()})")

Train 3051 | Val 652 | Test 654 (riesgo: 11)


## 3 · Augmentación de minoritarias (solo train)

In [3]:
def aumentar(t):
    V=[('bordes irregulares','márgenes irregulares'),('bordes espiculados','márgenes espiculados'),
       ('bordes mal definidos','límites imprecisos'),('imagen nodular','nódulo'),('nódulo','imagen nodular'),
       ('lesión nodular','nódulo'),('heterogéneamente densas','de densidad heterogénea'),
       ('muy densas','extremadamente densas'),('calcificaciones sospechosas','depósitos cálcicos sospechosos'),
       ('microcalcificaciones','calcificaciones puntiformes agrupadas'),('mama derecha','hemimama derecha'),
       ('mama izquierda','hemimama izquierda'),('se visualiza','se observa'),('se observa','se visualiza'),
       ('se evidencia','se observa')]
    s=t
    for o,r in random.sample(V,min(3,len(V))):
        if o in s: s=s.replace(o,r,1)
    return s
mask=np.isin(ytr,[3,4,5,6]); ta,la=[],[]
for txt,lab in zip(Xtr[mask],ytr[mask]):
    for _ in range(3): ta.append(aumentar(txt)); la.append(lab)
X_train_aug=np.concatenate([Xtr,np.array(ta)]); y_train_aug=np.concatenate([ytr,np.array(la)])
idx=np.random.RandomState(SEED).permutation(len(X_train_aug))
X_train_aug,y_train_aug=X_train_aug[idx],y_train_aug[idx]
cls_counts=np.array([ (y_train_aug==c).sum() for c in range(7) ],dtype=np.float64)
print("Train aumentado:", len(X_train_aug), "| conteo por clase:", cls_counts.astype(int))

Train aumentado: 3387 | conteo por clase: [ 676  418 1845  244  144   48   12]


## 4 · Funciones de pérdida: Logit Adjustment y LDAM

In [4]:
class LogitAdjustLoss(nn.Module):
    """Menon et al. 2021: suma tau*log(prior) a los logits en entrenamiento."""
    def __init__(self, cls_num_list, tau=1.0):
        super().__init__()
        p=np.array(cls_num_list,dtype=np.float64); p=p/p.sum()
        self.adj=torch.tensor(np.log(p+1e-12)*tau, dtype=torch.float32)
    def forward(self, x, target):
        return F.cross_entropy(x + self.adj.to(x.device), target)

class LDAMLoss(nn.Module):
    """Cao et al. 2019: margen por clase ∝ 1/n^¼, con escala s. weight para DRW."""
    def __init__(self, cls_num_list, max_m=0.5, s=30):
        super().__init__()
        m=1.0/np.sqrt(np.sqrt(np.array(cls_num_list,dtype=np.float64)))
        m=m*(max_m/np.max(m))
        self.m_list=torch.tensor(m,dtype=torch.float32); self.s=s; self.weight=None
    def forward(self, x, target):
        idx=torch.zeros_like(x,dtype=torch.bool); idx.scatter_(1,target.view(-1,1),True)
        batch_m=(idx.float()*self.m_list.to(x.device)[None,:]).sum(1,keepdim=True)
        out=torch.where(idx, x-batch_m, x)
        w=self.weight.to(x.device) if self.weight is not None else None
        return F.cross_entropy(self.s*out, target, weight=w)
print("Funciones de pérdida definidas.")

Funciones de pérdida definidas.


## 5 · Dataset y modelo

In [5]:
try: TOK=AutoTokenizer.from_pretrained(MODELO)
except Exception: MODELO=MODELO_ALT; TOK=AutoTokenizer.from_pretrained(MODELO)
class DS(Dataset):
    def __init__(self,t,l): self.t=list(t); self.l=list(l)
    def __len__(self): return len(self.t)
    def __getitem__(self,i):
        e=TOK(str(self.t[i]),truncation=True,padding='max_length',max_length=MAX_LENGTH,
              return_tensors='pt',return_token_type_ids=False)
        return {'input_ids':e['input_ids'].squeeze(0),'attention_mask':e['attention_mask'].squeeze(0),
                'labels':torch.tensor(self.l[i],dtype=torch.long)}
class Modelo(nn.Module):
    def __init__(self,m,n=7,dp=0.5):
        super().__init__(); self.enc=AutoModel.from_pretrained(m)
        self.dp=nn.Dropout(dp); self.cl=nn.Linear(self.enc.config.hidden_size,n)
    def forward(self,ids,mask):
        return self.cl(self.dp(self.enc(input_ids=ids,attention_mask=mask).last_hidden_state[:,0,:]))
print("Listo.")

Listo.


## 6 · Función de entrenamiento y evaluación

In [6]:
def entrenar_evaluar(metodo):
    print(f"\n{'='*58}\n  MÉTODO: {metodo}\n{'='*58}")
    tl=DataLoader(DS(X_train_aug,y_train_aug),batch_size=BATCH,shuffle=True)
    vl=DataLoader(DS(Xva,yva),batch_size=BATCH); tt=DataLoader(DS(Xte,yte),batch_size=BATCH)
    model=Modelo(MODELO).to(DEVICE)
    opt=torch.optim.AdamW(model.parameters(),lr=LR)
    sch=get_linear_schedule_with_warmup(opt,0,len(tl)*EPOCHS)

    # Pesos de clase para DRW (LDAM) o referencia
    pw=compute_class_weight('balanced',classes=np.unique(y_train_aug),y=y_train_aug)
    pv=np.ones(7,dtype=np.float32)
    for c,w in zip(np.unique(y_train_aug),pw): pv[c]=w
    peso_tensor=torch.tensor(pv).to(DEVICE)

    if metodo=='logit_adjustment':
        crit=LogitAdjustLoss(cls_counts, tau=1.0)
    elif metodo=='ldam':
        crit=LDAMLoss(cls_counts, max_m=0.5, s=30)   # DRW se activa en el bucle
    else:
        crit=nn.CrossEntropyLoss(weight=peso_tensor)  # baseline

    DRW_EPOCH=5  # LDAM: a partir de aquí se activa la reponderación
    def ep(loader,train=True):
        model.train() if train else model.eval(); tot=0; P=[]; R=[]
        with torch.set_grad_enabled(train):
            for j,b in enumerate(loader):
                ids=b['input_ids'].to(DEVICE); mk=b['attention_mask'].to(DEVICE); lb=b['labels'].to(DEVICE)
                if train: opt.zero_grad()
                logits=model(ids,mk); loss=crit(logits,lb)
                if train: loss.backward(); opt.step(); sch.step()
                tot+=loss.item(); P.extend(logits.argmax(1).cpu().numpy()); R.extend(lb.cpu().numpy())
                if train and (j+1)%80==0: print(f"   batch {j+1}/{len(loader)}",flush=True)
        return tot/len(loader), f1_score(R,P,average='macro',zero_division=0)

    print("Época | TrainF1 | ValF1")
    mejor,sin=0,0
    for e in range(1,EPOCHS+1):
        if metodo=='ldam' and e==DRW_EPOCH:
            crit.weight=peso_tensor; print(f"   [LDAM] DRW activado en época {e}")
        _,trf=ep(tl,True); _,vlf=ep(vl,False); m=""
        if vlf>mejor: mejor,sin=vlf,0; torch.save(model.state_dict(),f'mejor_{metodo}.pt'); m="  <-"
        else: sin+=1
        print(f"  {e:2d}  |  {trf:.4f} |  {vlf:.4f}{m}",flush=True)
        if sin>=PATIENCE: print(f"   Early stop en época {e}"); break

    model.load_state_dict(torch.load(f'mejor_{metodo}.pt',map_location=DEVICE)); model.eval()
    preds=[]; probs=[]; reales=[]
    with torch.no_grad():
        for b in tt:
            ids=b['input_ids'].to(DEVICE); mk=b['attention_mask'].to(DEVICE)
            pr=torch.softmax(model(ids,mk).float(),1).cpu().numpy()
            probs.extend(pr); preds.extend(pr.argmax(1)); reales.extend(b['labels'].numpy())
    preds=np.array(preds); reales=np.array(reales); probs=np.array(probs)
    f1m=f1_score(reales,preds,average='macro',zero_division=0)
    pr=probs[:,4:7].sum(1); yz=(reales>=4).astype(int); auc=roc_auc_score(yz,pr)
    yp=(pr>=0.5).astype(int); tn,fp,fn,tp=confusion_matrix(yz,yp,labels=[0,1]).ravel()
    sens=tp/(tp+fn) if (tp+fn) else 0; espec=tn/(tn+fp) if (tn+fp) else 0
    print(f"\n  >> F1-macro: {f1m:.4f} | AUC zona(deriv): {auc:.4f} | Sens: {sens:.3f} | Espec: {espec:.3f}")
    print(classification_report(reales,preds,target_names=[f'BI-RADS {i}' for i in range(7)],zero_division=0))
    return {'metodo':metodo,'f1':f1m,'auc':auc,'sens':sens,'espec':espec}

## 7 · Método 1 — Logit Adjustment

In [7]:
res_la = entrenar_evaluar('logit_adjustment')


  MÉTODO: logit_adjustment


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: PlanTL-GOB-ES/roberta-base-biomedical-clinical-es
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.decoder.bias      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.decoder.weight    | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Época | TrainF1 | ValF1
   batch 80/212
   batch 160/212
   1  |  0.3962 |  0.4424  <-
   batch 80/212
   batch 160/212
   2  |  0.5914 |  0.5280  <-
   batch 80/212
   batch 160/212
   3  |  0.7397 |  0.4873
   batch 80/212
   batch 160/212
   4  |  0.8342 |  0.4768
   batch 80/212
   batch 160/212
   5  |  0.9159 |  0.5280  <-
   batch 80/212
   batch 160/212
   6  |  0.9424 |  0.4757
   batch 80/212
   batch 160/212
   7  |  0.9651 |  0.4763
   batch 80/212
   batch 160/212
   8  |  0.9706 |  0.4820
   batch 80/212
   batch 160/212
   9  |  0.9694 |  0.4996
   Early stop en época 9

  >> F1-macro: 0.5865 | AUC zona(deriv): 0.8978 | Sens: 0.636 | Espec: 0.992
              precision    recall  f1-score   support

   BI-RADS 0       0.83      0.87      0.85       145
   BI-RADS 1       0.95      0.98      0.96        89
   BI-RADS 2       0.97      0.94      0.96       396
   BI-RADS 3       0.46      0.46      0.46        13
   BI-RADS 4       0.38      0.38      0.38         8
   BI

## 8 · Método 2 — LDAM (con DRW)

In [8]:
res_ldam = entrenar_evaluar('ldam')


  MÉTODO: ldam


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: PlanTL-GOB-ES/roberta-base-biomedical-clinical-es
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.decoder.bias      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.decoder.weight    | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Época | TrainF1 | ValF1
   batch 80/212
   batch 160/212
   1  |  0.3603 |  0.4433  <-
   batch 80/212
   batch 160/212
   2  |  0.5214 |  0.4714  <-
   batch 80/212
   batch 160/212
   3  |  0.6542 |  0.4464
   batch 80/212
   batch 160/212
   4  |  0.7858 |  0.5103  <-
   [LDAM] DRW activado en época 5
   batch 80/212
   batch 160/212
   5  |  0.7107 |  0.5661  <-
   batch 80/212
   batch 160/212
   6  |  0.8251 |  0.4761
   batch 80/212
   batch 160/212
   7  |  0.8829 |  0.4887
   batch 80/212
   batch 160/212
   8  |  0.9282 |  0.5473
   batch 80/212
   batch 160/212
   9  |  0.9500 |  0.4795
   Early stop en época 9

  >> F1-macro: 0.4747 | AUC zona(deriv): 0.8323 | Sens: 0.364 | Espec: 0.998
              precision    recall  f1-score   support

   BI-RADS 0       0.77      0.90      0.83       145
   BI-RADS 1       0.96      0.98      0.97        89
   BI-RADS 2       0.98      0.94      0.96       396
   BI-RADS 3       0.33      0.08      0.12        13
   BI-RADS 4       0.

## 9 · Comparación final contra el baseline

In [9]:
print("="*60)
print("COMPARACIÓN — auditor RoBERTa clínico (7 clases, test real)")
print("="*60)
print(f"{'Método':24} | {'F1-macro':>9} | {'AUC(d)':>7} | {'Sens':>6} | {'Espec':>6}")
print("-"*60)
print(f"{'Baseline (CE ponderada)':24} | {0.5871:>9.4f} | {0.93:>7.2f} | {0.636:>6.3f} | {0.97:>6.3f}")
for r in [res_la,res_ldam]:
    nom={'logit_adjustment':'Logit Adjustment','ldam':'LDAM + DRW'}[r['metodo']]
    print(f"{nom:24} | {r['f1']:>9.4f} | {r['auc']:>7.3f} | {r['sens']:>6.3f} | {r['espec']:>6.3f}")
print("="*60)
print("Nota: el AUC del baseline (0,93) viene del binario dedicado; el AUC(d) aquí es derivado de 7 clases (no comparable directo). La comparación limpia es el F1-macro.")

COMPARACIÓN — auditor RoBERTa clínico (7 clases, test real)
Método                   |  F1-macro |  AUC(d) |   Sens |  Espec
------------------------------------------------------------
Baseline (CE ponderada)  |    0.5871 |    0.93 |  0.636 |  0.970
Logit Adjustment         |    0.5865 |   0.898 |  0.636 |  0.992
LDAM + DRW               |    0.4747 |   0.832 |  0.364 |  0.998
Nota: el AUC del baseline (0,93) viene del binario dedicado; el AUC(d) aquí es derivado de 7 clases (no comparable directo). La comparación limpia es el F1-macro.


## 10 · Cómo interpretar

- **Compara el F1-macro** contra 0,5871 (es la comparación limpia, mismo modelo y métrica).
- Si alguno sube de forma clara (a 0,61+), ese método aporta y lo agregas como tu mejor configuración.
- Si quedan parecidos o por debajo, confirma que el límite no es el método de balanceo sino la cantidad de datos reales de riesgo, y reportas haber evaluado logit adjustment y LDAM además de los métodos previos.

**Parámetros que puedes ajustar** si quieres explorar más: `tau` en Logit Adjustment (1.0 por defecto; probar 0.5 o 1.5), y `max_m` / `s` / `DRW_EPOCH` en LDAM.

Recuerda: una sola corrida con clases tan pequeñas tiene varianza alta. Si un método parece mejor, confírmalo con 2-3 semillas antes de declararlo ganador.